# 2.4 · 通用模式：取消、进度、分页与日志

**Patterns: cancellation, progress, pagination and logging**

> **学完本课你能：** 实现并使用进度通知、带超时的取消、基于游标的分页和逐请求日志，并说明在日志功能被弃用之后应改用什么。

官方阅读材料：[Spec: Patterns overview](https://modelcontextprotocol.io/specification/2026-07-28/basic/patterns/index) · [Spec: Cancellation](https://modelcontextprotocol.io/specification/2026-07-28/basic/patterns/cancellation) · [Spec: Progress](https://modelcontextprotocol.io/specification/2026-07-28/basic/patterns/progress) · [Spec: Pagination](https://modelcontextprotocol.io/specification/2026-07-28/server/utilities/pagination) · [Spec: Logging](https://modelcontextprotocol.io/specification/2026-07-28/server/utilities/logging)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 通过 SDK 观察进度
`ctx.report_progress(progress, total, message)` 会发送 `notifications/progress`（进度通知），但<strong>只有</strong>在请求携带了 `progressToken`（进度令牌（Progress Token））时才会发送。`Client.call_tool(..., progress_callback=...)` 会替你创建令牌，并把通知转给你的回调函数。

In [ ]:
import json
import os
import sys
import tempfile
import textwrap
import time
import warnings
from pathlib import Path

import anyio
from mcp import Client, StdioServerParameters
from mcp.server import MCPServer
from mcp.server.mcpserver import Context
from mcp.shared.exceptions import MCPDeprecationWarning, MCPError

indexer = MCPServer("indexer")


@indexer.tool()
async def index_repo(files: int, ctx: Context) -> str:
    """Index a repository, reporting progress per file."""
    for i in range(1, files + 1):
        await anyio.sleep(0.05)
        await ctx.report_progress(i, files, f"indexed file {i}/{files}")
    return f"indexed {files} files"


async def show_progress(progress, total, message):
    bar = "#" * int(20 * progress / total)
    print(f"  [{bar:<20}] {progress:>4.0f}/{total:.0f}  {message}")


async with Client(indexer) as client:
    result = await client.call_tool("index_repo", {"files": 5}, progress_callback=show_progress)
    print(result.content[0].text)
    quiet = await client.call_tool("index_repo", {"files": 5})        # no callback -> no token -> no notifications
    print(quiet.content[0].text, "(no progressToken, so no notifications)")

### 2. 线路上的进度，以及一个检查规则的客户端
通过原始 stdio，你可以看到令牌（Token）是如何传递的。我们还让每条通知都经过一个小型校验器，检查规范中的规则：令牌必须已知且处于活跃状态，`progress` 必须严格递增。一个故意写错的工具会让进度倒退。

In [ ]:
import queue
import subprocess
import threading

workdir = Path(tempfile.mkdtemp())
(workdir / "progress_server.py").write_text(textwrap.dedent('''
    import anyio
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context

    mcp = MCPServer("progress-lab", log_level="ERROR")

    @mcp.tool()
    async def good(ctx: Context) -> str:
        """Well-behaved progress."""
        for i in (0.25, 0.5, 1.0):
            await ctx.report_progress(i, 1.0, f"{i:.0%}")
        return "ok"

    @mcp.tool()
    async def buggy(ctx: Context) -> str:
        """Progress that goes backwards."""
        for i in (30, 20, 40):
            await ctx.report_progress(i)
        return "ok"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))
ENVELOPE = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
            "io.modelcontextprotocol/clientCapabilities": {}}


def start(script, env=None):
    proc = subprocess.Popen([sys.executable, str(workdir / script)], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                            stderr=subprocess.DEVNULL, text=True, env=env)
    inbox = queue.Queue()
    threading.Thread(target=lambda: [inbox.put(l) for l in proc.stdout], daemon=True).start()
    return proc, inbox


def send(proc, message):
    proc.stdin.write(json.dumps(message) + "\n")
    proc.stdin.flush()


class ProgressTracker:
    def __init__(self):
        self.active = {}                                     # token -> last progress value

    def start(self, token):
        self.active[token] = None

    def check(self, params):
        token, value = params["progressToken"], params["progress"]
        if token not in self.active:
            return "VIOLATION: unknown or finished token"
        last = self.active[token]
        self.active[token] = value
        if last is not None and value <= last:
            return f"VIOLATION: progress went {last} -> {value} (MUST increase)"
        return "ok"

    def finish(self, token):
        self.active.pop(token, None)


proc, inbox = start("progress_server.py")
tracker = ProgressTracker()
for id_, tool in [(1, "good"), (2, "buggy")]:
    token = f"tok-{tool}"
    tracker.start(token)
    send(proc, {"jsonrpc": "2.0", "id": id_, "method": "tools/call",
                "params": {"name": tool, "arguments": {}, "_meta": {**ENVELOPE, "progressToken": token}}})
    while True:
        msg = json.loads(inbox.get(timeout=10))
        if msg.get("method") == "notifications/progress":
            print(f"  {json.dumps(msg['params']):<85} {tracker.check(msg['params'])}")
        else:
            tracker.finish(token)
            print(f"  response id={msg['id']} -> token {token!r} retired\n")
            break
proc.stdin.close()
_ = proc.wait(timeout=10)

### 3. 通过 SDK 取消：放弃、超时，以及服务器看到了什么
服务器把事件追加写入一个事件文件，这样我们就能从外部观察它。当调用方的 `anyio` 作用域放弃等待时，SDK 会在 stdio 上发送 `notifications/cancelled`，处理函数随即被中断。`read_timeout_seconds`（读取超时）的效果相同，并会抛出一个错误。

In [ ]:
(workdir / "cancel_server.py").write_text(textwrap.dedent('''
    import os
    import anyio
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context

    mcp = MCPServer("cancel-lab", log_level="ERROR")

    def note(text):
        with open(os.environ["EVENTS_FILE"], "a") as f:
            f.write(text + "\\n")

    @mcp.tool()
    async def crunch(seconds: float, ctx: Context) -> str:
        """A long job."""
        note(f"request {ctx.request_id}: started ({seconds}s of work)")
        try:
            await anyio.sleep(seconds)
        except BaseException as exc:
            note(f"request {ctx.request_id}: interrupted by {type(exc).__name__}, resources freed")
            raise
        note(f"request {ctx.request_id}: finished")
        return "crunched"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))
events_file = workdir / "events.log"
events_file.write_text("")
env = {**os.environ, "EVENTS_FILE": str(events_file)}
params = StdioServerParameters(command=sys.executable, args=[str(workdir / "cancel_server.py")], env=env)

async with Client(params) as client:
    with anyio.move_on_after(0.5) as scope:                 # the user pressed "stop" after 0.5 s
        await client.call_tool("crunch", {"seconds": 5})
    print("caller gave up:", scope.cancelled_caught)

    try:
        await client.call_tool("crunch", {"seconds": 5}, read_timeout_seconds=0.5)
    except MCPError as err:
        print(f"timeout -> MCPError {err.error.code}: {err.error.message}")

    print((await client.call_tool("crunch", {"seconds": 0.1})).content[0].text)
    await anyio.sleep(0.2)

print("\nserver-side events:")
print(events_file.read_text())

超时以错误码 `-32001` 呈现，它来自<strong>旧版、由实现自定义的</strong>范围。规范没有为纯本地错误分配错误码，所以 SDK 只好自己选一个。记住，这个错误是由你自己的客户端（Client）抛出的，而不是服务器发来的。

### 4. 手动取消，包括竞态情况
再次使用原始 stdio。我们取消一个正在运行的请求，然后发送两个<em>无效的</em>取消（一个已完成的 id 和一个未知的 id）。规范说服务器应当静默忽略它们，连接也会继续正常工作。

In [ ]:
proc, inbox = start("cancel_server.py", env=env)


def next_message(timeout=2.0):
    try:
        return json.loads(inbox.get(timeout=timeout))
    except queue.Empty:
        return None


send(proc, {"jsonrpc": "2.0", "id": "job-1", "method": "tools/call",
            "params": {"name": "crunch", "arguments": {"seconds": 10}, "_meta": ENVELOPE}})
time.sleep(1.0)                                              # let it start (process start-up + work)
send(proc, {"jsonrpc": "2.0", "method": "notifications/cancelled",
            "params": {"requestId": "job-1", "reason": "user pressed stop"}})
print("reply to job-1 after cancel:", next_message())

send(proc, {"jsonrpc": "2.0", "id": "job-2", "method": "tools/call",
            "params": {"name": "crunch", "arguments": {"seconds": 0.05}, "_meta": ENVELOPE}})
print("job-2:", next_message(5)["result"]["content"][0]["text"])
send(proc, {"jsonrpc": "2.0", "method": "notifications/cancelled", "params": {"requestId": "job-2"}})   # too late
send(proc, {"jsonrpc": "2.0", "method": "notifications/cancelled", "params": {"requestId": "never-sent"}})
print("after two invalid cancels:", next_message(1.0), "(silently ignored)")

send(proc, {"jsonrpc": "2.0", "id": "job-3", "method": "tools/call",
            "params": {"name": "crunch", "arguments": {"seconds": 0.05}, "_meta": ENVELOPE}})
print("job-3 still works:", next_message(5)["result"]["content"][0]["text"])
proc.stdin.close()
_ = proc.wait(timeout=10)
print("\nserver-side events:\n" + "\n".join(l for l in events_file.read_text().splitlines() if "job-" in l))

### 5. 看门狗：由进度重置的空闲超时，再加一个硬性上限
这里实现了规范中关于超时（Timeout）的建议。进度会延长<em>空闲</em>截止时间，但硬性上限即使对一个永远在报告进度的请求也会终止它。

In [ ]:
forever = MCPServer("forever")


@forever.tool()
async def stream_forever(ctx: Context) -> str:
    """Reports progress endlessly (a misbehaving server)."""
    i = 0
    while True:
        i += 1
        await ctx.report_progress(i, message=f"tick {i}")
        await anyio.sleep(0.2)


async def call_with_watchdog(client, name, args, idle=0.5, hard_max=1.5):
    started = last_progress = time.monotonic()

    async def on_progress(progress, total, message):
        nonlocal last_progress
        last_progress = time.monotonic()

    async with anyio.create_task_group() as tg:
        outcome = {}

        async def run():
            outcome["result"] = (await client.call_tool(name, args, progress_callback=on_progress)).content[0].text
            tg.cancel_scope.cancel()

        tg.start_soon(run)
        while True:
            await anyio.sleep(0.05)
            now = time.monotonic()
            if now - last_progress > idle:
                outcome["error"] = f"idle timeout ({idle}s without progress)"
            elif now - started > hard_max:
                outcome["error"] = f"hard maximum of {hard_max}s reached despite progress"
            if outcome:
                tg.cancel_scope.cancel()             # cancelling the call makes the SDK cancel the request
                break
    return outcome


async with Client(forever) as client:
    print(await call_with_watchdog(client, "stream_forever", {}))
async with Client(indexer) as client:
    print(await call_with_watchdog(client, "index_repo", {"files": 8}, idle=0.5, hard_max=5))

### 6. 使用不透明游标的分页
`MCPServer` 会把所有工具放在一页中返回。要实现分页（Pagination），请使用底层的 `Server` 并自己编写 `tools/list` 处理函数。这里的游标是一个客户端不得解释的 base64 数据块（不透明游标（Opaque Cursor）），错误的游标会得到 `-32602`。

In [ ]:
import base64

import mcp_types as types
from mcp.server import Server

CATALOG = [types.Tool(name=f"report_{i:02d}", description=f"Report number {i}", input_schema={"type": "object"})
           for i in range(7)]
PAGE_SIZE = 3                                                   # the SERVER decides


def make_cursor(offset: int) -> str:
    return base64.urlsafe_b64encode(json.dumps({"o": offset}).encode()).decode()


async def list_tools(ctx, params: types.PaginatedRequestParams | None) -> types.ListToolsResult:
    offset = 0
    if params is not None and params.cursor is not None:
        try:
            offset = json.loads(base64.urlsafe_b64decode(params.cursor))["o"]
        except Exception:
            raise MCPError(code=types.INVALID_PARAMS, message="Invalid cursor")
    page = CATALOG[offset:offset + PAGE_SIZE]
    more = offset + PAGE_SIZE < len(CATALOG)
    return types.ListToolsResult(tools=page, next_cursor=make_cursor(offset + PAGE_SIZE) if more else None)


pager = Server("pager", on_list_tools=list_tools)

async with Client(pager) as client:
    first = await client.list_tools()
    show = first.model_dump(by_alias=True, exclude_none=True, mode="json")
    print("first page on the wire:", json.dumps({k: v for k, v in show.items() if k != "tools"}),
          "| tools:", [t["name"] for t in show["tools"]])

    everything, cursor, pages = [], None, 0
    while True:                                                  # the loop every client should have
        page = await client.list_tools(cursor=cursor)
        pages += 1
        everything += [t.name for t in page.tools]
        if page.next_cursor is None:                             # only a MISSING cursor ends the loop
            break
        cursor = page.next_cursor
    print(f"collected {len(everything)} tools in {pages} pages:", everything)

    try:
        await client.list_tools(cursor="not-a-real-cursor")
    except MCPError as err:
        print("invalid cursor ->", err.error.code, err.error.message)

### 7. 日志：已弃用，但仍在线路上
客户端通过设置 `io.modelcontextprotocol/logLevel`（逐请求日志级别）来<strong>逐请求</strong>选择开启日志（Logging）。SDK 的 `Client(log_level=..., logging_callback=...)` 会在每个请求上加上这个键。没有级别时，服务器必须保持沉默。SDK 用 `MCPDeprecationWarning` 标记了整个特性。

In [ ]:
chatty = MCPServer("chatty")


@chatty.tool()
async def migrate(ctx: Context) -> str:
    """Run a migration, logging along the way."""
    await ctx.debug("opening connection pool")
    await ctx.info({"step": "migrate", "tables": 12})
    await ctx.warning("table 'legacy_users' has no primary key")
    return "migrated"


async def print_log(params):
    print(f"   log [{params.level:<7}] {params.data}")


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", MCPDeprecationWarning)
    for level in ["debug", "warning", None]:
        print(f"log_level={level!r}:")
        async with Client(chatty, log_level=level, logging_callback=print_log) as client:
            print("  ", (await client.call_tool("migrate", {})).content[0].text)
print("\ndeprecation warnings raised:", sorted({str(w.message) for w in caught}))

在原始线路上，级别只是 `_meta` 中的一个键，未知的级别会被拒绝：

In [ ]:
(workdir / "log_server.py").write_text(textwrap.dedent('''
    import warnings
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context
    warnings.simplefilter("ignore")
    mcp = MCPServer("log-lab", log_level="ERROR")

    @mcp.tool()
    async def migrate(ctx: Context) -> str:
        """Run a migration."""
        await ctx.info("migration started")
        return "migrated"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))
proc, inbox = start("log_server.py")
for id_, level in [(1, "info"), (2, "loud")]:
    send(proc, {"jsonrpc": "2.0", "id": id_, "method": "tools/call",
                "params": {"name": "migrate", "arguments": {},
                           "_meta": {**ENVELOPE, "io.modelcontextprotocol/logLevel": level}}})
    while True:
        msg = json.loads(inbox.get(timeout=10))
        print(f"logLevel={level!r}:", json.dumps(msg)[:150])
        if "id" in msg:
            break
send(proc, {"jsonrpc": "2.0", "id": 3, "method": "server/discover", "params": {"_meta": ENVELOPE}})
caps = json.loads(inbox.get(timeout=10))["result"]["capabilities"]
print("\ndeclared capabilities:", sorted(caps), "(the spec says log emitters MUST declare 'logging')")
proc.stdin.close()
_ = proc.wait(timeout=10)

注意最后一行：这个 SDK 版本会发出 `notifications/message`，却没有在 `server/discover` 中声明 `logging` 能力（Capability）。请把能力列表当作提示，并处理实际到达的内容。

### 8. 替代方案：stderr 上的结构化日志
对于 stdio 服务器，迁移路径就是 stderr。这里服务器每行向 stderr 写一个 JSON 对象，宿主（Host）把它与协议流分开捕获。不需要选择开启，也不需要声明能力，而且没有任何内容会进入模型的上下文。

In [ ]:
(workdir / "stderr_server.py").write_text(textwrap.dedent('''
    import json, logging, sys, time
    from mcp.server import MCPServer

    class JsonLines(logging.Formatter):
        def format(self, record):
            return json.dumps({"ts": round(time.time(), 3), "level": record.levelname.lower(),
                               "logger": record.name, "msg": record.getMessage()})

    handler = logging.StreamHandler(sys.stderr)          # never stdout on stdio!
    handler.setFormatter(JsonLines())
    log = logging.getLogger("billing")
    log.addHandler(handler)
    log.setLevel(logging.INFO)
    log.propagate = False

    mcp = MCPServer("billing", log_level="ERROR")

    @mcp.tool()
    def charge(cents: int) -> str:
        """Charge a (pretend) card."""
        log.info("charge requested: %d cents", cents)
        if cents > 10_000:
            log.warning("large charge flagged for review")
        return "charged"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))
proc = subprocess.Popen([sys.executable, str(workdir / "stderr_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
send(proc, {"jsonrpc": "2.0", "id": 1, "method": "tools/call",
            "params": {"name": "charge", "arguments": {"cents": 25_000}, "_meta": ENVELOPE}})
print("stdout (protocol):", proc.stdout.readline().strip()[:110], "...")
proc.stdin.close()
_ = proc.wait(timeout=10)
print("stderr (logs the host can capture):")
for line in proc.stderr.read().splitlines():
    if line.startswith("{"):
        print("  ", json.loads(line))

对于远程服务器，对应的方案是 OpenTelemetry。客户端在 `_meta` 中放入 `traceparent`（第 2.1 课），服务器把 span 和日志导出到你的可观测性后端，并与智能体（Agent）的追踪相关联。

### 动手练习
1. 让 `index_repo` 以 `total=None`（大小未知）报告进度（Progress）。修改 `show_progress`，让它仍然能显示一些有用的东西。哪个字段仍然必须递增？
2. 修改 `call_with_watchdog`，让它在超时时还打印收到了多少条进度通知。然后对 `index_repo` 试试 `idle=0.1`：出了什么问题？在生产环境中你会如何选择空闲超时值？
3. 让分页器在最后一页返回 `next_cursor=""` 而不是 `None`。第 6 节中的客户端循环会终止吗？规范说客户端必须如何对待 `""`？bug 出在哪一方？
4. 像第 8 节那样，把第 7 节中的 `migrate` 工具改写为向 stderr 写日志，并列出两类绝不能出现在任何一种日志中的数据。